# SimPro-ValueNetwork — Value Network Inference

Set the project path, REPRESENTATION, TOKENIZATION, and DEPTH, then run cells in order. Bundles for all four representations are included.

Extract the project to `/content/SimPro-ValueNetwork`, or set `PROJECT_DIR` to your project root. Google Drive is optional: enable `USE_GOOGLE_DRIVE` and select your mounted project path. Checkpoints may be stored at any readable location.


The project is SimPro-ValueNetwork; the distribution/import name is simpro_vn, with simpro-vn and simpro-vn-train console entrypoints. Use normal installation in a fresh runtime. Use [Comprehensive Validation](colab_comprehensive_validation.ipynb) for all functional checks in one run.


In [ ]:
USE_GOOGLE_DRIVE = False  # Enable only if your files are stored in Google Drive.
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import sys
PROJECT_DIR = Path("/content/SimPro-ValueNetwork")
REPRESENTATION = "smiles"  # smiles, smarts, safe, selfies
WEIGHTS_DIR = PROJECT_DIR / "weights" / REPRESENTATION  # Set any readable checkpoint directory.
TOKENIZATION = 5000  # 5000, 10000, 20000, 50000, or "no_bpe"
DEPTH = 6  # 2, 4, or 6; match the checkpoint depth.
stem = f"{REPRESENTATION}_no_bpe" if TOKENIZATION == "no_bpe" else f"{REPRESENTATION}_bpe_min_freq_{TOKENIZATION}"
CHECKPOINT_PATH = WEIGHTS_DIR / f"value_network_best_{stem}_d{DEPTH}.pt"
# Inputs must already use the selected representation; these are simple interface examples.
sample_texts = {
    "smiles": ["CCO", "c1ccccc1"],
    "smarts": ["[#6]-[#6]-[#8]", "[#6]"],
    "safe": ["CCO", "CC"],
    "selfies": ["[C][C][O]", "[C][C]"],
}[REPRESENTATION]
sample_prefix = {
    "smiles": "CC", "smarts": "[#6]-[#6]",
    "safe": "CC", "selfies": "[C][C]",
}[REPRESENTATION]


In [ ]:
%pip install "{PROJECT_DIR}[chemistry]"


In [ ]:
from simpro_vn import ValueNetworkPredictor

predictor = ValueNetworkPredictor(
    checkpoint_path=CHECKPOINT_PATH,
    representation=REPRESENTATION,
    min_frequency=TOKENIZATION,
    depth=DEPTH,
    device="auto",
    # Optional: paths to content-equivalent copies of the selected preprocessing assets.
)


In [ ]:
predictions = predictor.predict(sample_texts)
predictions


In [ ]:
prefix_prediction = predictor.predict(sample_prefix, complete=False)
prefix_prediction


In [ ]:
trajectory = predictor.predict_trajectory(sample_texts[0])
list(zip(trajectory["tokens"], trajectory["predictions"]))


---

## KV-cache during generation

This example consumes IDs from one fixed encoding. A generator may supply its own IDs only when it uses exactly the same tokenizer.


In [ ]:
session = predictor.new_session()
for token_id in predictor.encode(sample_texts[0]):
    prediction = session.step(token_id)[-1]
    print(token_id, prediction)
session.reset()


## Optional: Descriptions and Tensors

Dictionary prediction remains available. Other PyTorch programs may use these interfaces to retain outputs on the predictor device in property units, or z-scores with standardized=True. Value Network gradients are not tracked. See [MODEL_INTERFACE.md](../docs/MODEL_INTERFACE.md) for the contract.


In [ ]:
import torch

print(predictor.describe())
values = predictor.predict_tensor(sample_texts)
ids = torch.tensor([predictor.encode(sample_texts[0])])
from_ids = predictor.predict_ids(ids)
session = predictor.new_session()
incremental = session.step_tensor(ids[0])
torch.testing.assert_close(values[:1], from_ids)
torch.testing.assert_close(incremental[-1:], from_ids)
print('Tensor shape:', values.shape, '| device:', values.device)
session.reset()

## Training and Fine-Tuning

Use [colab_training_validation.ipynb](colab_comprehensive_validation.ipynb) for training checks, [colab_finetuning.ipynb](colab_finetuning.ipynb) for user-data fine-tuning, and [colab_training.ipynb](colab_training.ipynb) for scratch/curriculum training.
